# PerioVision AI: train the models on a free Colab GPU

**How to use this notebook**
1. `Runtime` → `Change runtime type` → **T4 GPU** → Save.
2. `Runtime` → **Run all**. When asked, allow access to Google Drive (results are saved there so nothing is lost if Colab disconnects).
3. Leave the tab open. Part A (tooth detector) takes roughly 2–4 hours on a free T4.
4. When it finishes, download the folder `MyDrive/PerioVision/export/` and give it to Claude (or follow the last cell).

**What it trains**
- **Part A: tooth detector with FDI numbers (11–48)** on the public **DENTEX** panoramic dataset (MICCAI 2023 challenge, 634 panoramic X-rays with per-tooth FDI boxes, licence CC BY-NC-SA 4.0: academic, non-commercial use with attribution). Data is downloaded *inside Colab*; no patient data touches your laptop or the GitHub repo.
- **Part B (optional): CEJ / root-apex / bone-crest keypoints.** No open dataset with these labels can be downloaded without an access request (BoneLoss-PAN769, perio-KPT and DenPAR are all gated). Once you have one, put it in Drive in YOLO-pose format and Part B trains on it. Part B skips itself otherwise.

**Honesty rules built in**
- The data is split once, with a fixed seed, into train / val / **test**. The test images are never used for training or model selection; the final numbers come only from them.
- Horizontal flip augmentation is **off**: flipping a panoramic X-ray swaps left and right teeth and would silently corrupt the FDI labels.
- If resuming after a disconnect, just Run all again: finished steps are skipped and training resumes from the last checkpoint in Drive.

In [ ]:
# 0. Settings (you can leave these as they are)
MODEL_SIZE   = "yolo11m"   # yolo11s = faster, yolo11l = more accurate but slower. m is the best fit for a free T4.
IMGSZ        = 1280         # panoramic X-rays are wide; teeth need resolution
EPOCHS       = 150
PATIENCE     = 30           # stop early if validation stops improving for 30 epochs
BATCH        = -1           # -1 = pick the largest batch that fits the GPU
SEED         = 42
POSE_DATASET = "/content/drive/MyDrive/PerioVision/pose_dataset"   # Part B input (optional)

In [ ]:
# 1. Check the GPU and mount Google Drive
import subprocess, os
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU: Runtime > Change runtime type > T4 GPU")
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/PerioVision"
for d in ("runs", "export", "splits"):
    os.makedirs(f"{ROOT}/{d}", exist_ok=True)
print("Saving results to", ROOT)

In [ ]:
# 2. Install the same Ultralytics version the PerioVision backend uses
!pip -q install "ultralytics==8.4.21" "huggingface_hub>=0.23"
import ultralytics; ultralytics.checks()

## Part A: tooth detector (FDI)

In [ ]:
# 3. Download DENTEX (training_data.zip is ~11 GB; this takes a few minutes inside Google's network)
import os, zipfile, glob
from huggingface_hub import hf_hub_download
RAW = "/content/dentex"
os.makedirs(RAW, exist_ok=True)
if not glob.glob(f"{RAW}/**/*quadrant_enumeration*.json", recursive=True):
    z = hf_hub_download("ibrahimhamamci/DENTEX", "DENTEX/training_data.zip", repo_type="dataset", local_dir=RAW)
    with zipfile.ZipFile(z) as f:
        # only the fully labelled tooth-enumeration subset is needed
        members = [m for m in f.namelist() if "quadrant_enumeration/" in m and "disease" not in m]
        f.extractall(RAW, members)
    os.remove(z)
json_path = [p for p in glob.glob(f"{RAW}/**/*.json", recursive=True) if "quadrant_enumeration" in os.path.basename(p) and "disease" not in p][0]
img_dir = os.path.join(os.path.dirname(json_path), "xrays")
print(json_path, "|", len(os.listdir(img_dir)), "images")

In [ ]:
# 4. Convert DENTEX (COCO JSON: quadrant + tooth number) to YOLO format with FDI class names,
#    and make one fixed train / val / test split that is saved to Drive.
import json, random, shutil
from collections import Counter
coco = json.load(open(json_path))
q_name = {c["id"]: str(c["name"]) for c in coco["categories_1"]}
t_name = {c["id"]: str(c["name"]) for c in coco["categories_2"]}
FDI = [f"{q}{n}" for q in (1, 2, 3, 4) for n in range(1, 9)]
cls_of = {k: i for i, k in enumerate(FDI)}
print("quadrant names:", sorted(set(q_name.values())), " tooth names:", sorted(set(t_name.values())))

imgs = {im["id"]: im for im in coco["images"]}
per_img, skipped = {}, Counter()
for a in coco["annotations"]:
    fdi = q_name[a["category_id_1"]] + t_name[a["category_id_2"]]
    if fdi not in cls_of:
        skipped[fdi] += 1; continue
    im = imgs[a["image_id"]]; W, H = im["width"], im["height"]
    x, y, w, h = a["bbox"]
    per_img.setdefault(im["id"], []).append(f"{cls_of[fdi]} {(x+w/2)/W:.6f} {(y+h/2)/H:.6f} {w/W:.6f} {h/H:.6f}")
print("labelled images:", len(per_img), " boxes:", sum(map(len, per_img.values())), " skipped labels:", dict(skipped))

split_file = f"{ROOT}/splits/dentex_split.json"
if os.path.exists(split_file):
    split = json.load(open(split_file))           # reuse the same split after a disconnect
else:
    names = sorted(imgs[i]["file_name"] for i in per_img)
    random.Random(SEED).shuffle(names)
    n = len(names); n_test = n_val = round(n * 0.10)
    split = {"test": names[:n_test], "val": names[n_test:n_test+n_val], "train": names[n_test+n_val:]}
    json.dump(split, open(split_file, "w"), indent=1)
print({k: len(v) for k, v in split.items()})

DET = "/content/det"
by_name = {imgs[i]["file_name"]: i for i in per_img}
for part, files in split.items():
    os.makedirs(f"{DET}/images/{part}", exist_ok=True); os.makedirs(f"{DET}/labels/{part}", exist_ok=True)
    for fn in files:
        dst = f"{DET}/images/{part}/{fn}"
        if not os.path.exists(dst):
            shutil.copy(os.path.join(img_dir, fn), dst)
        open(f"{DET}/labels/{part}/{os.path.splitext(fn)[0]}.txt", "w").write("\n".join(per_img[by_name[fn]]) + "\n")
yaml = f"path: {DET}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n" + "".join(f"  {i}: '{k}'\n" for i, k in enumerate(FDI))
open(f"{DET}/dentex_fdi.yaml", "w").write(yaml)
print(yaml[:200], "...")

In [ ]:
# 5. Train (resumes automatically from Drive if Colab disconnected)
from ultralytics import YOLO
run_dir = f"{ROOT}/runs/detector"
last, done = f"{run_dir}/weights/last.pt", f"{run_dir}/DONE"
if os.path.exists(done):
    print("Detector already trained, skipping.")
elif os.path.exists(last):
    YOLO(last).train(resume=True)
else:
    YOLO(f"{MODEL_SIZE}.pt").train(
        data=f"{DET}/dentex_fdi.yaml", imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH,
        project=f"{ROOT}/runs", name="detector", exist_ok=True, seed=SEED, deterministic=True,
        fliplr=0.0,              # NEVER flip: it swaps left/right teeth and breaks FDI labels
        degrees=3, translate=0.05, scale=0.25, mosaic=1.0, close_mosaic=15,
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.3,   # radiographs are greyscale: vary brightness only
        cos_lr=True, optimizer="AdamW", lr0=0.001, cache="disk", workers=2, plots=True)
open(done, "w").write("ok")

In [ ]:
# 6. Honest evaluation on the held-out TEST split (never seen during training)
import json
best = f"{ROOT}/runs/detector/weights/best.pt"
m = YOLO(best).val(data=f"{DET}/dentex_fdi.yaml", split="test", imgsz=IMGSZ, batch=4, plots=True,
                   project=f"{ROOT}/runs", name="detector_test", exist_ok=True)
det_metrics = {
    "dataset": "DENTEX quadrant_enumeration (CC BY-NC-SA 4.0), fixed seed split",
    "split_sizes": {k: len(v) for k, v in split.items()},
    "model": MODEL_SIZE, "imgsz": IMGSZ,
    "test_mAP50": round(float(m.box.map50), 4), "test_mAP50_95": round(float(m.box.map), 4),
    "test_precision": round(float(m.box.mp), 4), "test_recall": round(float(m.box.mr), 4),
    "per_class_mAP50_95": {FDI[i]: round(float(v), 4) for i, v in zip(m.box.ap_class_index, m.box.maps[m.box.ap_class_index])},
}
json.dump(det_metrics, open(f"{ROOT}/export/detector_test_metrics.json", "w"), indent=2)
print(json.dumps({k: v for k, v in det_metrics.items() if k != "per_class_mAP50_95"}, indent=2))

In [ ]:
# 7. Tooth-level check the app cares about: how many teeth per X-ray are found with the RIGHT FDI number
import numpy as np, glob
model = YOLO(best)
tp = fp = fn = 0
for img in sorted(glob.glob(f"{DET}/images/test/*")):
    lab = f"{DET}/labels/test/{os.path.splitext(os.path.basename(img))[0]}.txt"
    gt = {}
    for line in open(lab):
        c, cx, cy, w, h = map(float, line.split()); gt[int(c)] = (cx-w/2, cy-h/2, cx+w/2, cy+h/2)
    r = model.predict(img, imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    pred = {}
    for c, b, s in zip(r.boxes.cls.int().tolist(), r.boxes.xyxyn.tolist(), r.boxes.conf.tolist()):
        if c not in pred or s > pred[c][1]: pred[c] = (b, s)
    def iou(a, b):
        ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
        u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - ix*iy
        return ix*iy/u if u else 0
    for c, (b, _) in pred.items():
        if c in gt and iou(b, gt[c]) >= 0.5: tp += 1
        else: fp += 1
    fn += sum(1 for c in gt if c not in pred or iou(pred[c][0], gt[c]) < 0.5)
tooth = {"tooth_level_precision": round(tp/(tp+fp), 4), "tooth_level_recall": round(tp/(tp+fn), 4),
         "tooth_level_F1": round(2*tp/(2*tp+fp+fn), 4), "conf": 0.25, "iou": 0.5}
det_metrics.update(tooth); json.dump(det_metrics, open(f"{ROOT}/export/detector_test_metrics.json", "w"), indent=2)
shutil.copy(best, f"{ROOT}/export/dental_yolov8n.pt")   # file name the PerioVision backend expects
print(tooth, "\nSaved export/dental_yolov8n.pt")

## Part B (optional): CEJ / apex / bone-crest keypoints

Needs a keypoint dataset in Drive at `MyDrive/PerioVision/pose_dataset/` in YOLO-pose format:
```
pose_dataset/images/{train,val,test}/*.png
pose_dataset/labels/{train,val,test}/*.txt   # class cx cy w h  cej_x cej_y v  apex_x apex_y v  crest_x crest_y v
```
Keypoint order must be **CEJ, root apex, bone crest** (the backend reads them in that order).
Where to get real labels: request access to [BoneLoss-PAN769](https://zenodo.org/records/21939261) (CC BY 4.0) or [perio-KPT](https://zenodo.org/records/17272200) (CC BY-NC-SA, university access), or annotate ~150 of the DENTEX test images yourself in CVAT / Roboflow (free) with those three points per tooth.
If the folder is missing, this part is skipped.

In [ ]:
# 8. Train and test the keypoint model (skipped if no pose dataset is in Drive)
if not os.path.isdir(f"{POSE_DATASET}/images/train"):
    print("No pose dataset at", POSE_DATASET, "- Part B skipped. Part A results are complete.")
else:
    has_test = os.path.isdir(f"{POSE_DATASET}/images/test")
    open("/content/pose.yaml", "w").write(
        f"path: {POSE_DATASET}\ntrain: images/train\nval: images/val\n" + ("test: images/test\n" if has_test else "") +
        "kpt_shape: [3, 3]\nflip_idx: [0, 1, 2]\nnames:\n  0: tooth\n")
    prun = f"{ROOT}/runs/landmarks"
    if os.path.exists(f"{prun}/DONE"):
        print("Landmark model already trained, skipping.")
    elif os.path.exists(f"{prun}/weights/last.pt"):
        YOLO(f"{prun}/weights/last.pt").train(resume=True)
    else:
        YOLO(f"{MODEL_SIZE}-pose.pt").train(
            data="/content/pose.yaml", imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH,
            project=f"{ROOT}/runs", name="landmarks", exist_ok=True, seed=SEED, fliplr=0.0,
            degrees=3, scale=0.25, hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, cos_lr=True, workers=2)
    open(f"{prun}/DONE", "w").write("ok")
    pbest = f"{prun}/weights/best.pt"
    pm = YOLO(pbest).val(data="/content/pose.yaml", split="test" if has_test else "val", imgsz=IMGSZ, batch=4)
    pose_metrics = {"split": "test" if has_test else "val (no test split supplied!)",
                    "box_mAP50": round(float(pm.box.map50), 4), "pose_mAP50": round(float(pm.pose.map50), 4),
                    "pose_mAP50_95": round(float(pm.pose.map), 4)}
    json.dump(pose_metrics, open(f"{ROOT}/export/landmark_test_metrics.json", "w"), indent=2)
    shutil.copy(pbest, f"{ROOT}/export/dental_landmark_yolov8n-pose.pt")
    print(pose_metrics)

## Done: bring the models back

Download `MyDrive/PerioVision/export/` (right-click the folder in Drive → Download). It contains:
- `dental_yolov8n.pt`: the new tooth detector (named so the app picks it up)
- `detector_test_metrics.json`: its honest test-set numbers
- `dental_landmark_yolov8n-pose.pt` and `landmark_test_metrics.json` (only if Part B ran)

Then on your laptop, from the project folder:
```powershell
.\run.ps1 install-models -From "$HOME\Downloads\export"
```
That backs up the old weights, copies the new ones in, signs them and runs the tests.